In [0]:
# ==========================================
# SILVER LAYER - PREMIER LEAGUE STANDINGS
# ==========================================

# Read the Bronze Delta table
bronze_df = spark.table(
    "workspace.default.bronze_standings"
)

# Inspect the JSON structure
bronze_df.printSchema()

# Display the top-level fields
print("Columns:", bronze_df.columns)

# Check how many standings groups are available
from pyspark.sql.functions import size, col

bronze_df.select(
    size(col("standings")).alias("standings_groups")
).show()

# Display the nested standings data
display(bronze_df.select("standings"))


In [0]:

from pyspark.sql.functions import col, explode

# 1. Read the Bronze standings table
bronze_df = spark.table("workspace.default.bronze_standings")

# 2. Extract standings groups
standings_df = bronze_df.select(
    col("season.id").alias("season_id"),
    explode(col("standings")).alias("standing")
)

# 3. Extract individual team records
teams_df = standings_df.select(
    col("season_id"),
    col("standing.type").alias("standing_type"),
    col("standing.stage").alias("stage"),
    explode(col("standing.table")).alias("team")
)

# 4. Create the Silver DataFrame
silver_df = teams_df.select(
    col("season_id"),
    col("standing_type"),
    col("stage"),
    col("team.team.id").alias("team_id"),
    col("team.team.name").alias("team_name"),
    col("team.team.tla").alias("team_code"),
    col("team.position").alias("position"),
    col("team.playedGames").alias("played"),
    col("team.won").alias("won"),
    col("team.draw").alias("drawn"),
    col("team.lost").alias("lost"),
    col("team.goalsFor").alias("goals_for"),
    col("team.goalsAgainst").alias("goals_against"),
    col("team.goalDifference").alias("goal_difference"),
    col("team.points").alias("points"),
    col("team.form").alias("form")
)

print("Silver DataFrame created!")
print("Rows:", silver_df.count())
display(silver_df)


In [0]:
from pyspark.sql.functions import col, explode

# ==========================================
# 4. DATA QUALITY CHECKS
# ==========================================

# Remove rows missing essential identifiers
silver_df = silver_df.dropna(
    subset=["season_id", "team_id", "standing_type"]
)

# Remove duplicate records
silver_df = silver_df.dropDuplicates(
    ["season_id", "standing_type", "stage", "team_id"]
)

# Check for duplicate team records
total_rows = silver_df.count()

unique_rows = silver_df.select(
    "season_id", "standing_type", "stage", "team_id"
).distinct().count()

assert total_rows == unique_rows, "Duplicate standings records found!"

# Check for missing points
assert silver_df.filter(
    col("points").isNull()
).count() == 0, "Missing points detected!"

# Check goal difference calculation
invalid_goal_difference = silver_df.filter(
    col("goal_difference") !=
    (col("goals_for") - col("goals_against"))
).count()

assert invalid_goal_difference == 0, "Invalid goal difference detected!"

print("Standings validation passed!")
print("Total clean standings records:", total_rows)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_standings")
)

print("Silver standings table created successfully!")


In [0]:
%sql
SELECT
    position,
    team_name,
    played,
    won,
    drawn,
    lost,
    points
FROM workspace.default.silver_standings
WHERE standing_type = 'TOTAL'
ORDER BY position;